# Evidently: is the data still like what the model learned from?

Great Expectations asks **"is this data broken?"** Evidently asks two different questions:

| Question | Needs outcomes? | Available |
|---|---|---|
| **Data drift:** are the inputs distributed differently from the reference? | no | as soon as a batch arrives |
| **Prediction drift:** are the model's risk scores distributed differently? | no | as soon as a batch is scored |
| **Performance:** is the model still right? | **yes** | ~90 days later, when outcomes are known |

The model is the 2024 logistic regression; new months arrive in 2025, some with the
simulator's problem scenarios. We'll see which problems each kind of monitoring catches,
and which it misses.

In [ ]:
import sys
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")
sys.path.insert(0, "../03-mlflow")        # reuse the MLflow lesson's helpers
import emr_model as em

import pandas as pd
from evidently import BinaryClassification, DataDefinition, Dataset, Report
from evidently.metrics import DriftedColumnsCount, MissingValueCount, RocAuc, ValueDrift, MeanValue
from evidently.presets import DataDriftPreset
from evidently.tests import lte
from sklearn.metrics import roc_auc_score

pd.set_option("display.width", 200, "display.max_columns", 30, "display.precision", 3)
DATA = Path("data")
Path("reports").mkdir(exist_ok=True)

em.simulate("2024-01", 12, out=DATA / "raw")                 # training year
history = em.load_months("2024-01", "2024-12", raw=DATA / "raw")
model = em.build_model("logistic").fit(history[em.FEATURES], history[em.LABEL])
history["risk"] = model.predict_proba(history[em.FEATURES])[:, 1]


def scored(month, scenario="baseline"):
    """One new month of visits (from a given scenario), with the model's risk scores."""
    folder = DATA / scenario
    em.simulate(month, 1, scenario=scenario, out=folder)
    df = em.load_months(month, month, raw=folder)
    df["risk"] = model.predict_proba(df[em.FEATURES])[:, 1]
    return df

print(f"model trained on {len(history)} visits from 2024")

## 1. A first drift report, and a false alarm

Evidently compares a **current** batch with a **reference** batch, column by column.
Start naively: reference = all of 2024, current = January 2025 (a normal month).

In [ ]:
num_all = em.NUMERIC + ["risk"]                     # numeric inputs + the model's risk score
definition = DataDefinition(numerical_columns=num_all, categorical_columns=em.CATEGORICAL)
cols = num_all + em.CATEGORICAL

january = scored("2025-01")
report = Report([DataDriftPreset()])                              # one drift test per column
snapshot = report.run(Dataset.from_pandas(january[cols], data_definition=definition),     # current
                      Dataset.from_pandas(history[cols], data_definition=definition))     # reference
snapshot.save_html("reports/01_naive_january.html")


def drift_table(snapshot):
    """Per-column drift scores from a report, as a DataFrame."""
    rows = [{"column": m["config"]["column"], "method": m["config"]["method"],
             "score": m["value"], "drifted": m["value"] >= m["config"]["threshold"]}
            for m in snapshot.dict()["metrics"] if m["config"]["type"].endswith("ValueDrift")]
    return pd.DataFrame(rows).set_index("column").sort_values("score", ascending=False)

drift_table(snapshot).head(6)

Open `reports/01_naive_january.html` in a browser for the full interactive report.

**How drift is measured.** For each column, Evidently compares the two distributions:
numeric columns with a normalised *Wasserstein distance*, categorical ones with a
*Jensen–Shannon distance*. Above **0.1** the column counts as drifted. (Evidently picks
the method by column type and size; both can be changed.)

**Two columns "drifted" in a perfectly normal month.** `lean_season` is 0 in January but
1 in a third of the 2024 reference (June–September), and the model's risk is higher in
the lean season. That's **seasonality, not drift**: the reference was the wrong
comparison. Lesson one: **choose the reference to match the season**, and leave out
calendar columns that differ by construction.

In [ ]:
LEAN = {6, 7, 8, 9}
num = [c for c in em.NUMERIC if c != "lean_season"] + ["risk"]       # drop the calendar column
definition = DataDefinition(numerical_columns=num, categorical_columns=em.CATEGORICAL)
cols = num + em.CATEGORICAL


def reference_for(month):
    """Same-season reference: 2024's lean months for a lean month, the other months otherwise."""
    lean = int(month[5:7]) in LEAN
    return history[history["lean_season"] == int(lean)]


def drift_report(current, month):
    return Report([DataDriftPreset()]).run(
        Dataset.from_pandas(current[cols], data_definition=definition),
        Dataset.from_pandas(reference_for(month)[cols], data_definition=definition))

drift_table(drift_report(january, "2025-01")).head(4)

With a same-season reference, the normal month shows **no drift**. Now the scenarios.

## 2. What drift detection catches, and what it misses

In [ ]:
scenarios = ["baseline", "new_regions", "muac_mm", "missing_anthro", "concept_shift", "messy_export"]
summary = {}
for sc in scenarios:
    cur = scored("2025-01", sc)
    snap = drift_report(cur, "2025-01")
    snap.save_html(f"reports/02_{sc}.html")
    table = drift_table(snap)
    dataset = next(m for m in snap.dict()["metrics"] if "DriftedColumnsCount" in m["metric_name"])["value"]
    summary[sc] = {
        "drifted columns": ", ".join(table.index[table["drifted"]]) or "-",
        "risk drift": table.loc["risk", "score"],
        "share drifted": dataset["share"],
        "mean predicted risk": cur["risk"].mean(),
        "actual GAM rate (known 90 days later)": cur[em.LABEL].mean(),
    }
pd.DataFrame(summary).T

Read it row by row (reference: mean predicted risk 9%, GAM rate ~9%):

- **baseline**: nothing drifts. Good: no false alarms.
- **new_regions**: `region`, `water_source_minutes` and **`risk`** drift; mean risk rises
  to 13%. Covariate drift, caught. But only 3 of 19 columns drifted: Evidently's default
  *dataset-level* verdict (drift if ≥ 50% of columns drift) says **no dataset drift**.
  Never rely on that default; decide which columns matter (section 3).
- **muac_mm**: `muac_cm` drifts enormously and the mean risk collapses to ~1%: the model
  thinks no child is at risk. Drift catches it, but this is a *broken* batch:
  Great Expectations should have stopped it before scoring.
- **missing_anthro**: 70% of weights and heights are missing, yet **only `risk`** drifts.
  Drift tests compare the values that are present; they don't count the missing ones.
  Missing values need their own check (section 3).
- **concept_shift**: **nothing drifts**. Inputs and predictions look normal (mean risk
  9%), but the real GAM rate is ~20%: the model is badly under-predicting. Only
  outcomes reveal this (section 4).
- **messy_export**: `muac_cm`, `risk` drift: the Amhara mm values. Again GX's job first.

**`risk` is the single most useful signal**: it drifts in four of the five problem
scenarios, and it needs no outcomes.

## 3. From reports to a monitoring gate

Reports are for people. For a pipeline you want **tests** that pass or fail. Evidently
attaches tests to metrics. Our gate, written down in advance:

- predicted risk must not drift (score ≤ 0.1),
- the columns we know matter most (`region`, `muac_cm`, `whz`) must not drift,
- at most 5% missing weights and heights.

In [ ]:
def monitoring_gate(current, month):
    report = Report([
        ValueDrift(column="risk", tests=[lte(0.1)]),
        ValueDrift(column="region", tests=[lte(0.1)]),
        ValueDrift(column="muac_cm", tests=[lte(0.1)]),
        ValueDrift(column="whz", tests=[lte(0.1)]),
        MissingValueCount(column="weight_kg", share_tests=[lte(0.05)]),   # share, not count
        MissingValueCount(column="height_cm", share_tests=[lte(0.05)]),
    ])
    snap = report.run(Dataset.from_pandas(current[cols], data_definition=definition),
                      Dataset.from_pandas(reference_for(month)[cols], data_definition=definition))
    failed = [t["name"] for t in snap.dict()["tests"] if str(t["status"]).endswith("FAIL")]
    return snap, failed

for sc in scenarios:
    _, failed = monitoring_gate(scored("2025-01", sc), "2025-01")
    print(f"{sc:15s} {'ALERT' if failed else 'ok   '}  {failed}")

Every problem scenario now raises an alert, **except `concept_shift`**: nothing about
the inputs or predictions changed. The world did.

## 4. Performance monitoring, once outcomes arrive

About 90 days after each visit we know who developed GAM. Then we can check the model
itself. Tell Evidently which column is the outcome and which is the predicted probability:

In [ ]:
perf_definition = DataDefinition(
    numerical_columns=["risk"],
    classification=[BinaryClassification(target=em.LABEL, prediction_probas="risk")],
)
perf_cols = ["risk", em.LABEL]


def performance(current, month):
    snap = Report([RocAuc(), MeanValue(column="risk"), MeanValue(column=em.LABEL)]).run(
        Dataset.from_pandas(current[perf_cols], data_definition=perf_definition),
        Dataset.from_pandas(reference_for(month)[perf_cols], data_definition=perf_definition))
    auc, predicted, observed = [m["value"] for m in snap.dict()["metrics"]]
    return {"roc_auc": auc, "predicted GAM rate": predicted, "observed GAM rate": observed,
            "observed / predicted": observed / predicted}

pd.DataFrame({sc: performance(scored("2025-01", sc), "2025-01")
              for sc in ["baseline", "concept_shift"]}).T

Under concept shift the model still **ranks** children well (ROC-AUC doesn't drop: 0.82),
but it predicts a 9% GAM rate when the real rate is ~20%: **observed / predicted
≈ 2**. Its risk numbers are now far too low. A clinic using a fixed risk cut-off would
miss most of the new cases; one flagging the top 20% would still flag the right children
but far fewer than now need follow-up. This is the signal to retrain: exactly the
challenger story from the MLflow registry notebook.

The catch: this signal arrives **~90 days late**. Prediction drift is the early warning;
performance monitoring is the confirmation.

## 5. A monitoring log for 2025

Put it together: each month, run the gate on the new batch; once outcomes arrive,
add performance. A plausible first half of 2025:

In [ ]:
timeline = {"2025-01": "baseline", "2025-02": "baseline", "2025-03": "new_regions",
            "2025-04": "muac_mm", "2025-05": "concept_shift", "2025-06": "concept_shift"}
log = []
for month, scenario in timeline.items():
    current = scored(month, scenario)
    snap, failed = monitoring_gate(current, month)
    snap.save_html(f"reports/05_monitoring_{month}.html")
    perf = performance(current, month)
    log.append({"month": month, "(hidden truth)": scenario,
                "gate": "ALERT" if failed else "ok",
                "failed checks": "; ".join(f.split(":")[0] for f in failed) or "-",
                "mean risk": current["risk"].mean(),
                "observed/predicted (+90 days)": perf["observed / predicted"]})
pd.DataFrame(log).set_index("month")

- **March:** the gate flags the population change on the day the data arrives.
- **April:** the unit bug is flagged (and in the full pipeline GX blocks it first).
  Three months later, observed/predicted ≈ 6 confirms the scores were useless that month.
- **May–June:** the gate stays quiet; only the outcomes, three months later, show the
  model under-predicting by about half. June is lean season: the same-season reference
  keeps that from looking like drift.

This table is what the **Prefect** pipeline will produce every month, and what decides
whether to retrain (MLflow challenger), investigate the data (GX, the EMR team), or
check fairness again (Fairlearn).

### What to remember
- Drift = **different**, not **wrong**. Choose the reference carefully (season!).
- Watch **predicted risk** first: no outcomes needed, catches most problems.
- Drift doesn't see **missing values**; check them explicitly.
- Don't trust default verdicts (50% of columns); write your own gate.
- **Concept drift is invisible** without outcomes, and outcomes come late.